# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3896


In [1]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface/hub/"
print(f"Setting cache path to {CACHE_PATH}")

os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Code formatting and linting

# !black notebooks/Llama-3-8B-quant.ipynb
# !pylint notebooks/Llama-3-8B-quant.ipynb

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface/hub/
MemTotal: 1007.72 GB
MemFree: 12.67 GB
MemAvailable: 916.35 GB
Free GPU Memory (GB): 39.3896

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################



/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/hub/token
Login successful
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################

Free GPU Memory (GB): 39.3896. Context: Warm up notebook.


## 2. Loading Model

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda"
model_name = "ISTA-DASLab/Meta-Llama-3-8B-AQLM-PV-1Bit-1x16"

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map=device)
model.NAME = model_name

tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

if tokenizer.model_max_length > 1e6:
  print(f"Tokenizer model max length reduced from {tokenizer.model_max_length} to 2048 to fit in memory")
  tokenizer.model_max_length = 2048

!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
print(f"Loaded model {model_name} with the following configuration:")
print(f"- model max length: {tokenizer.model_max_length}")
print(f"- dtype: {model.dtype}")
print(f"- device: {model.device}")
print(f"- parameters: {(lambda p: f'{p / 1e9:.1f}B' if p > 1e9 else (f'{p / 1e6:.1f}M' if p > 1e6 else str(p)))(model.num_parameters())}")
print(f"- memory footprint: {model.get_memory_footprint() / (1024 ** 3):.2f} GB")
print(f"- vocabulary size: {tokenizer.vocab_size}")
print(f"- padding token ID: {tokenizer.pad_token_id}")
print(f"- special tokens: {tokenizer.special_tokens_map}")

from src.evaluations.evaluate_memory import record_gpu_memory
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load model")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory
Free GPU Memory (GB): 35.1523
Loaded model ISTA-DASLab/Meta-Llama-3-8B-AQLM-PV-1Bit-1x16 with the following configuration:
- model max length: 2048
- dtype: torch.float16
- device: cuda:0
- parameters: 2.0B
- memory footprint: 3.80 GB
- vocabulary size: 128000
- padding token ID: 128001
- special tokens: {'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>', 'pad_token': '<|end_of_text|>'}
Free GPU Memory (GB): 35.1523. Context: Load model.


In [5]:
model.__dict__

{'training': False,
 '_parameters': OrderedDict(),
 '_buffers': OrderedDict(),
 '_non_persistent_buffers_set': set(),
 '_backward_pre_hooks': OrderedDict(),
 '_backward_hooks': OrderedDict(),
 '_is_full_backward_hook': None,
 '_forward_hooks': OrderedDict(),
 '_forward_hooks_with_kwargs': OrderedDict(),
 '_forward_hooks_always_called': OrderedDict(),
 '_forward_pre_hooks': OrderedDict(),
 '_forward_pre_hooks_with_kwargs': OrderedDict(),
 '_state_dict_hooks': OrderedDict(),
 '_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_post_hooks': OrderedDict(),
 '_modules': OrderedDict([('model',
               LlamaModel(
                 (embed_tokens): Embedding(128256, 4096)
                 (layers): ModuleList(
                   (0-31): 32 x LlamaDecoderLayer(
                     (self_attn): LlamaSdpaAttention(
                       (q_proj): QuantizedLinear()
                       (k_proj): QuantizedLinear()
                      

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda"

# model_name = "EleutherAI/gpt-neo-125m"  # Lightweight model for debugging purposes
# model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "openai-community/gpt2-large"

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
# TODO: Check why dtype = auto solved the problem
# TODO: what is the default value of torch_dtype -> look in the githubb documentation
# Always use "auto"
model.NAME = model_name

tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

if tokenizer.model_max_length > 1e6:
  print(f"Tokenizer model max length reduced from {tokenizer.model_max_length} to 2048 to fit in memory")
  tokenizer.model_max_length = 2048

!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
print(f"Loaded model {model_name} with the following configuration:")
print(f"- model max length: {tokenizer.model_max_length}")
print(f"- dtype: {model.dtype}")
print(f"- device: {model.device}")
print(f"- parameters: {(lambda p: f'{p / 1e9:.1f}B' if p > 1e9 else (f'{p / 1e6:.1f}M' if p > 1e6 else str(p)))(model.num_parameters())}")
print(f"- memory footprint: {model.get_memory_footprint() / (1024 ** 3):.2f} GB")
print(f"- vocabulary size: {tokenizer.vocab_size}")
print(f"- padding token ID: {tokenizer.pad_token_id}")
print(f"- special tokens: {tokenizer.special_tokens_map}")

from src.evaluations.evaluate_memory import record_gpu_memory
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load model")

Free GPU Memory (GB): 36.8418
Loaded model TinyLlama/TinyLlama-1.1B-Chat-v1.0 with the following configuration:
- model max length: 2048
- dtype: torch.bfloat16
- device: cuda:0
- parameters: 1.1B
- memory footprint: 2.05 GB
- vocabulary size: 32000
- padding token ID: 2
- special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}
Free GPU Memory (GB): 36.8418. Context: Load model.


In [ ]:
# Example inference

from transformers import AutoTokenizer
import transformers 
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)
pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    torch_dtype="auto",
    device_map="auto",
)

prompt = "What famous tower is in Paris?"
formatted_prompt = (
    f"### Human: {prompt}### Assistant:"
)

sequences = pipeline(
    formatted_prompt,
    do_sample=True,
    top_k=50,
    top_p = 0.7,
    num_return_sequences=1,
    repetition_penalty=1.1,
    max_new_tokens=500,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")


## 3. Loading Datasets

### 3.1. WikiText

In [4]:
# Initialize the datamodule
import os
from src.data.WikiTextDataModule import WikiTextDataModule

print("\n################################")
print("Setting up WikiTextDataModule...")
print("################################\n")

wikitext_data_module = WikiTextDataModule(
  directory_dataset=os.getcwd(),
  batch_size=1,
  sequence_length=2048,
  tokenizer_name=model_name,
  seed=3,
)

wikitext_dataloader = wikitext_data_module.test_dataloader()

print("\n################################")
print("Printing properties of WikiTextDataModule...")
print("################################\n")

# Print properties
print(f"Length of train dataset: {len(wikitext_data_module.train_dataset)}")
print(f"Length of validation dataset: {len(wikitext_data_module.val_dataset)}")
print(f"Length of test dataset: {len(wikitext_data_module.test_dataset)}")

print("\nTotal number of tokens in each dataset:")
print(f"Train dataset: {sum([len(data_string) for data_string in wikitext_data_module.train_dataset['text']])}")
print(f"Validation dataset: {sum([len(data_string) for data_string in wikitext_data_module.val_dataset['text']])}")
print(f"Test dataset: {sum([len(data_string) for data_string in wikitext_data_module.test_dataset['text']])}")

total_string = "".join([data_string for data_string in wikitext_data_module.val_dataset['text']])
total_string_len = len(total_string)
tokenized_string = tokenizer.encode(total_string, return_tensors="pt")

print(f"\nLength of total validation dataset (characters): {total_string_len}")
print(f"Length of tokenized validation dataset (tokens): {len(tokenized_string[0])}")
print(f"Tokenizer compression rate: {(100 * len(tokenized_string[0]) / total_string_len):.2f}%")

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset
dataset_size = len(wikitext_dataloader)
print(f"\nNumber of batches in validation dataloader: {dataset_size}")

for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 1:
        print(f"\nBatch {i + 1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}...")  # Print the first 500 characters
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")


################################
Setting up WikiTextDataModule...
################################



Token indices sequence length is longer than the specified maximum sequence length for this model (341469 > 2048). Running this sequence through the model will result in indexing errors



################################
Printing properties of WikiTextDataModule...
################################

Length of train dataset: 36718
Length of validation dataset: 3760
Length of test dataset: 4358

Total number of tokens in each dataset:
Train dataset: 10892990
Validation dataset: 1142150
Test dataset: 1285622


Token indices sequence length is longer than the specified maximum sequence length for this model (292446 > 2048). Running this sequence through the model will result in indexing errors



Length of total validation dataset (characters): 1142150
Length of tokenized validation dataset (tokens): 292446
Tokenizer compression rate: 25.60%

Number of batches in validation dataloader: 666

Batch 1:
  Original Text: 1807 , which had a major negative effect on the state 's large merchant fleet . Gore was in 1808 elected to the Massachusetts House of Representatives , where he successfully led Federalist efforts to ensure the selection of a Federalist slate of presidential electors . He also spearheaded actions to drive Senator John Quincy Adams from the Federalist Party over his support of Thomas Jefferson 's foreign policy . The legislature elected Adams ' successor nine months early , and gave Adams suffic...
  Input data (first 5 tokens): tensor([29871, 29896, 29947, 29900, 29955])
  Target labels (first 5 tokens): tensor([29896, 29947, 29900, 29955,  1919])
  Input data shape: torch.Size([1, 512])
  Target labels shape: torch.Size([1, 512])


### 3.2. OpenAssistant

In [ ]:
# Initialize the datamodule
import os
from src.data.OpenAssistantDataModule import OpenAssistantDataModule

# Data Module
oasst_data_module = OpenAssistantDataModule(
  directory_dataset=os.getcwd(),
  batch_size=1,
  sequence_length=2048,
  tokenizer_name=model_name,
  seed=1
)

# Data Loader
# oasst_dataloader = oasst_data_module.train_dataloader()
oasst_dataloader = oasst_data_module.val_dataloader()

print(f"Length of datasets:", len(oasst_data_module.train_dataset), len(oasst_data_module.val_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

oasst_dataset_size = len(oasst_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")
for i, (data, target) in enumerate(oasst_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

### 3.3 RedPajama

In [4]:
from datasets import load_dataset

ds = load_dataset("togethercomputer/RedPajama-Data-1T-Sample")

: 

## 4. Quantization

### 4.4 AQLM

## 5. Evaluation

In [6]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Free GPU Memory (GB): 4.53906


### 5.1. Perplexity

In [8]:
import torch
import torchmetrics
import tqdm
from torch.cuda.amp import autocast

# Evaluate Perplexity
print("\n################################")
print("Evaluating Perplexity...")
print("################################\n")

def evaluate_perplexity(model, dataloader, device="cuda", to_device=False):
    if isinstance(model, torch.nn.Module):
        model.eval()
        print(f"Model in evaluation mode. Device: {device}")
    metric = torchmetrics.text.Perplexity(ignore_index=-100).to(device)  # -100 is the padding token.

    for i, (x, y) in enumerate(dataloader):
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)
        
        with torch.no_grad():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
            
            # Metric on current batch
            perplexity = metric(logits.float(), y)
            print(f"Perplexity: {perplexity:.2f}")

    # Metric on all batches using custom accumulation
    perplexity = metric.compute()
    print(f"\nFinal Perplexity (PPL): {perplexity:.3f}")
    return perplexity.item()


################################
Evaluating Perplexity...
################################



In [9]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Model in evaluation mode. Device: cuda
Processing batch 0
Free GPU Memory (GB): 3.92773
Perplexity: 5.55
Processing batch 1
Free GPU Memory (GB): 3.60156
Perplexity: 9.71
Processing batch 2
Free GPU Memory (GB): 3.63086
Perplexity: 12.43
Processing batch 3
Free GPU Memory (GB): 3.63477
Perplexity: 11.15
Processing batch 4
Free GPU Memory (GB): 3.63477
Perplexity: 5.95
Processing batch 5
Free GPU Memory (GB): 3.63477
Perplexity: 7.38
Processing batch 6
Free GPU Memory (GB): 3.60156
Perplexity: 5.51
Processing batch 7
Free GPU Memory (GB): 3.60156
Perplexity: 4.95
Processing batch 8
Free GPU Memory (GB): 3.60156
Perplexity: 7.85
Processing batch 9
Free GPU Memory (GB): 3.59375
Perplexity: 8.51
Processing batch 10
Free GPU Memory (GB): 3.63477
Perplexity: 8.89
Processing batch 11
Free GPU Memory (GB): 3.63477
Perplexity: 7.77
Processing batch 12
Free GPU Memory (GB): 3.63477
Perplexity: 6.97
Processing batch 13
Free GPU Memory (GB): 3.63477
Perplexity: 8.06
Processing batch 14
Free GPU Me

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(quantized_model, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_same, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_dynamic, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
import numpy as np
lls = torch.tensor(lls)
print(stride)
print(lls/stride)
print(torch.exp(lls / (stride)))
print(torch.exp(lls.sum() / (31 * stride)))

ppls = [ppl for ppl in ppls]
print(ppls)

print(xs[2])
print(ys[2])
print(input_ids_list[2])
print(target_ids_list[2])

print(outputs[0])
print()

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
evaluate_perplexity(model_bnb_8bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(awq_model, wikitext_dataloader, device="cuda")

In [ ]:
list_of_models = [model, model_bnb_8bit, model_bnb_4bit]
results = {}
for model in list_of_models:
  print(f"Perplexity for model {model.NAME}: {evaluate_perplexity(model_bnb_8bit, wikitext_data_module, device)}"

# Print perplexity results
#print(f"Perplexity (8-bit): {perplexity_8bit:.4f}")
#print(f"Perplexity (4-bit): {perplexity_4bit:.4f}")
print(f"Perplexity (Original): {perplexity_original:.4f}")

### 5.2. Brier Score

In [ ]:
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast

class BrierScore:
    def __init__(self, device="cpu"):
        self.device = device
        self.reset()

    def reset(self):
        self.total_brier_score = 0.0
        self.num_batches = 0

    def update(self, probs, targets):
        brier_score = torch.mean((probs - targets) ** 2)
        self.total_brier_score += brier_score.item()
        self.num_batches += 1

    def compute(self):
        if self.num_batches == 0:
            return 0.0
        return self.total_brier_score / self.num_batches

def evaluate_brier_score(model, dataloader, device="cuda", to_device=False):
    if to_device:
        model.to(device)

    if isinstance(model, torch.nn.Module):
        model.eval()

    print(f"Model in evaluation mode. Device: {device}")
    
    # Initialize BrierScore metric
    metric = BrierScore(device=device)
    
    for i, (x, y) in enumerate(dataloader):
        if i > 10:
            break
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)

        with torch.no_grad() and autocast():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

            # Shift logits and target_ids to the left by 1 for calculating the Brier score
            shifted_logits = logits[:, :-1].contiguous()
            shifted_target_ids = x[:, 1:].contiguous()

            # Flatten the logits and target_ids for calculation
            shifted_logits = shifted_logits.view(-1, shifted_logits.size(-1))
            shifted_target_ids = shifted_target_ids.view(-1)

            # Filter out the -100 targets
            valid_indices = shifted_target_ids != -100
            valid_logits = shifted_logits[valid_indices]
            valid_target_ids = shifted_target_ids[valid_indices]

            # Get the probabilities
            probs = F.softmax(valid_logits, dim=-1)

            # Create one-hot target vectors
            targets = F.one_hot(valid_target_ids, num_classes=probs.size(-1)).float()

            # Update the metric with the current batch's results
            metric.update(probs, targets)

    # Compute the final Brier score across all batches
    avg_brier_score = metric.compute()
    print(f"Final Brier Score: {avg_brier_score:.10f}")

    return avg_brier_score

# Assuming wikitext_data_module and model are defined elsewhere
wikitext_dataloader = wikitext_data_module.test_dataloader()
final_brier_score = evaluate_brier_score(model, wikitext_dataloader, device=device)
print(f"\nFinal Brier Score: {final_brier_score:.10f}")

In [ ]:
evaluate_brier_score(model, tokenizer, wikitext_dataloader, factor=100, device=device)

In [ ]:
evaluate_brier_score(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)